# AnswerCheck — 01 Colab setup

Run this notebook at the start of **every** Colab session. It:

1. downloads (clones) the code from GitHub the first time, or updates it (pulls) later,
2. installs the pinned packages and the `answercheck` package,
3. prints the GPU and package versions,
4. sets the random seed,
5. runs the tests to confirm everything works.

**Before you start:** Runtime > Change runtime type > **T4 GPU** (free). The setup works without a GPU too, but training later will be slow.

Colab does **not** save code changes back to GitHub. Edit code on your own computer, push it, then re-run this notebook to pull the changes.

## 1. Get the code (clone first time, pull after)

In [ ]:
import os

REPO_URL = "https://github.com/hassnain45/answercheck.git"  # public repo: no token needed
REPO_DIR = "/content/answercheck"

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!git log -1 --oneline

## 2. Install packages

We install everything from `requirements.txt` **except `torch`**. Colab already ships a PyTorch build that matches its GPU drivers; replacing it would download about 2-3 GB every session and could break GPU support. The exact Colab torch version is printed in step 3 and will be saved with every run.

If Colab shows a **"Restart session"** button after this cell, click it, then continue from step 3 (do not re-run step 2).

In [ ]:
# %pip (not !pip) installs into the Python this notebook is running on.
!grep -v -E '^torch==' requirements.txt > /tmp/requirements-colab.txt
%pip install -q -r /tmp/requirements-colab.txt
%pip install -q -e . --no-deps

## 3. Import the package and print the environment

In [ ]:
import sys

# An editable install only becomes visible to an already-running notebook after a
# restart, so we also add the src/ folder to the import path directly.
SRC_DIR = "/content/answercheck/src"
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

import answercheck
from answercheck.environment import print_environment_report

print("imported answercheck from", answercheck.__file__)
env = print_environment_report()

## 4. Set the random seed

In [ ]:
from answercheck.training.seeds import set_seed

SEED = 42
set_seed(SEED)
print("seed set to", SEED)

## 5. Run the tests

Phase 0 is done on Colab when this prints `passed` with no failures.

In [ ]:
import sys

# sys.executable = the same Python this notebook uses.
!{sys.executable} -m pytest -q